<a href="https://colab.research.google.com/github/syedmahmoodiagents/NLP/blob/main/Simple_RNN_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

In [1]:
corpus = [
    "I love machine learning lot",
    "word2vec is a great algorithm",
    "Implementing word2vec is really fun"
]

In [2]:

sentences = [s.lower().split() for s in corpus]

In [3]:
sentences

[['i', 'love', 'machine', 'learning', 'lot'],
 ['word2vec', 'is', 'a', 'great', 'algorithm'],
 ['implementing', 'word2vec', 'is', 'really', 'fun']]

In [5]:
vocab = sorted(set(word for sent in sentences for word in sent))

In [6]:
vocab

['a',
 'algorithm',
 'fun',
 'great',
 'i',
 'implementing',
 'is',
 'learning',
 'lot',
 'love',
 'machine',
 'really',
 'word2vec']

In [7]:
word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for i,w in enumerate(vocab)}

In [8]:
idx2word

{0: 'a',
 1: 'algorithm',
 2: 'fun',
 3: 'great',
 4: 'i',
 5: 'implementing',
 6: 'is',
 7: 'learning',
 8: 'lot',
 9: 'love',
 10: 'machine',
 11: 'really',
 12: 'word2vec'}

In [9]:
vocab_size = len(vocab)

In [10]:
word2idx

{'a': 0,
 'algorithm': 1,
 'fun': 2,
 'great': 3,
 'i': 4,
 'implementing': 5,
 'is': 6,
 'learning': 7,
 'lot': 8,
 'love': 9,
 'machine': 10,
 'really': 11,
 'word2vec': 12}

In [13]:
X = []
Y = []
for sent in sentences:
    for i in range(len(sent)-1):
        X.append(word2idx[sent[i]])
        Y.append(word2idx[sent[i+1]])

In [14]:
X = torch.tensor(X)
Y = torch.tensor(Y)

In [15]:
print(X)
print(Y)

tensor([ 4,  9, 10,  7, 12,  6,  0,  3,  5, 12,  6, 11])
tensor([ 9, 10,  7,  8,  6,  0,  3,  1, 12,  6, 11,  2])


In [16]:

X = X.reshape(1, 12)

In [29]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hx=None):
        x = self.embedding(x)
        out, hx = self.rnn(x, hx) # [1, 12, 16], [1, 1, 16]
        out = out.reshape(-1, out.shape[2]) # [12, 16] # for next word
        out = self.fc(out)
        return out

In [30]:
mod = SimpleRNN(13)

In [20]:
l_fn = nn.CrossEntropyLoss()
optimiz = optim.Adam(mod.parameters(), lr=0.01)

In [21]:
for epoch in range(300):
    optimiz.zero_grad()
    Yp = mod(X)
    ls = l_fn(Yp, Y)
    ls.backward()
    optimiz.step()

    if epoch % 50 == 0:
        print("Epoch:", epoch, "Loss:", ls.item())

Epoch: 0 Loss: 2.718787431716919
Epoch: 50 Loss: 0.1387508660554886
Epoch: 100 Loss: 0.04484471306204796
Epoch: 150 Loss: 0.024067165330052376
Epoch: 200 Loss: 0.015743592754006386
Epoch: 250 Loss: 0.011202242225408554


In [24]:
def pred_next(word):
    mod.eval()
    idx = torch.tensor([[word2idx[word.lower()]]])

    with torch.no_grad():
        out = mod(idx)
        pred = torch.argmax(out, dim=1).item()

    return idx2word[pred]

In [28]:
pred_next("machine")

'learning'

# Now lets try to do it multiple sentences

In [32]:
X.shape

torch.Size([1, 12])

In [33]:
XX = X.reshape(3,4)

In [34]:
XX

tensor([[ 4,  9, 10,  7],
        [12,  6,  0,  3],
        [ 5, 12,  6, 11]])

In [36]:
YY = Y.reshape(3,4)

In [59]:
YY

tensor([[ 9, 10,  7,  8],
        [ 6,  0,  3,  1],
        [12,  6, 11,  2]])

In [37]:
vocab_size

13

In [48]:

class NextWordRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hx=None):
        x = self.embedding(x) # [3, 4, 100]
        out, hx = self.rnn(x, hx) # [3, 4, 16], [1, 3, 16]
        out = out.reshape(-1, out.shape[2]) # [12, 16] # for next word
        out = self.fc(out) # [12, 13]
        return out

In [49]:
model = NextWordRNN(13)

In [50]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [69]:
YY.reshape(-1)

tensor([ 9, 10,  7,  8,  6,  0,  3,  1, 12,  6, 11,  2])

In [65]:
for epoch in range(300):
    optimizer.zero_grad()
    outputs = model(XX)
    loss = loss_fn(outputs, YY.reshape(-1)) # flattenning YY making it equivalent to Y
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print("Epoch:", epoch, "Loss:", loss.item())

Epoch: 0 Loss: 2.607501745223999
Epoch: 50 Loss: 0.15978719294071198
Epoch: 100 Loss: 0.13305126130580902
Epoch: 150 Loss: 0.12614001333713531
Epoch: 200 Loss: 0.12281578034162521
Epoch: 250 Loss: 0.12091321498155594


In [66]:
def predict_next(word):
    model.eval()
    idx = torch.tensor([[word2idx[word.lower()]]])

    with torch.no_grad():
        out = model(idx)
        pred = torch.argmax(out, dim=1).item()

    return idx2word[pred]

In [73]:
print("machine=>", predict_next("machine"))
print("is=>", predict_next("is"))
print("word2vec=>", predict_next("word2vec"))

machine=> learning
is=> really
word2vec=> is
